In [51]:
!pip install -q git+https://github.com/zhouy185/rosa-starter.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [52]:
import numpy as np
from starter import ZONES, TIME_BLOCKS, COSTS, PROMISE, delivery_times
import math # use later for integer

In [53]:
times = delivery_times("Far West", "Fri/Sat eve", 45)
print(len(times), "orders; average delivery time", round(times.mean(), 1),
"minutes")

210 orders; average delivery time 43.9 minutes


# **Part I a**

I combined the selected orders and calculated the percentage delivered after the promise. With a 45-minute promise, the overall late rate is 8.29%.

In [54]:
def late_p (zone,time_blocks,promise): # select late_p using zone time blocks and promise
  all_times =[]  # give a list to save order delivery times
  if zone == "all" and time_blocks == "all":  # for all zone and time blocks
    for z in ZONES:
      for tb in TIME_BLOCKS:
        all_times.append(delivery_times(z,tb,promise,1))

  elif zone =="all":
    for z in ZONES:
      all_times.append(delivery_times(z,time_blocks,promise,1)) # for all zones but specific time blocks

  elif time_blocks =="all":
    for tb in TIME_BLOCKS:
      all_times.append(delivery_times(zone,tb,promise,1)) # for all time blocks but specific zone

  else:
    all_times.append(delivery_times(zone,time_blocks,promise,1)) # for both specific zone and time blocks

  all_times =np.concatenate(all_times) # flatten all times into a single alray

  return (all_times > promise) .mean()*100. # give a result as in %

In [55]:
print(f"{late_p("all","all",PROMISE)}%") # show result for all zone and all time blocks in %

8.291873963515755%


In [56]:
print(f"{late_p("all","Lunch",PROMISE)}%") # show result for all zone during lunch time in %

3.3333333333333335%


In [57]:
print(f"{late_p("Far West","all",PROMISE)}%") # show result for far west zone during all time blocks in %

15.606936416184972%


In [58]:
print(f"{late_p("Far West","Lunch",PROMISE)}%") # show result for far west zone during lunch time in %

2.5%


# **Part I b**

I sorted all 12 combinations by late rate.

In [59]:
late_sort = []  # Create an empty list for sort result
for z in ZONES:
    for tb in TIME_BLOCKS:
      rate = late_p(z, tb,PROMISE) # Call late_p to calculate the late percentage for the current combination
      late_sort.append((z,tb,rate)) # for rate add to list

def get_late_rate(item):  # get the rate from list
  return item[2]

late_sort.sort(key=get_late_rate,reverse=True) # Sort the saved results by rate from highest to lowest.

for item in late_sort:
  print(f"{item[0]} | {item[1]} | {item[2]}%") # show sort list

Far West | Fri/Sat eve | 36.666666666666664%
North | Fri/Sat eve | 16.666666666666664%
Far West | Weekday eve | 11.176470588235295%
Central | Fri/Sat eve | 6.842105263157896%
North | Weekday eve | 6.0%
Far West | Other | 4.6875%
Central | Weekday eve | 4.242424242424243%
North | Lunch | 3.684210526315789%
Central | Lunch | 3.4615384615384617%
Central | Other | 2.631578947368421%
North | Other | 2.5%
Far West | Lunch | 2.5%


# **Part I c**

I combined the selected delivery times and calculated their mean. The overall average is 31.19 minutes.

In [60]:
def average_time(zone, time_blocks, promise): # select average time using zone time blocks and promise
  all_times =[]  # give a list to save order delivery times
  if zone == "all" and time_blocks == "all":  # for all zone and time blocks
    for z in ZONES:
      for tb in TIME_BLOCKS:
        all_times.append(delivery_times(z,tb,promise,1))

  elif zone =="all":
    for z in ZONES:
      all_times.append(delivery_times(z,time_blocks,promise,1)) # for all zones but specific time blocks

  elif time_blocks =="all":
    for tb in TIME_BLOCKS:
      all_times.append(delivery_times(zone,tb,promise,1)) # for all time blocks but specific zone

  else:
    all_times.append(delivery_times(zone,time_blocks,promise,1)) # for both specific zone and time blocks

  all_times =np.concatenate(all_times) # flatten all times into a single alray

  return all_times.mean() # Return average delivery time in minutes

In [61]:
print(f"Far West | Lunch | {average_time('Far West', 'Lunch', PROMISE)} minutes") # Specified zones + Specified Time

Far West | Lunch | 30.58083333333333 minutes


In [62]:
print(f"All zones | Lunch | {average_time('all', 'Lunch', PROMISE)} minutes")  # all zones+ Specified Time

All zones | Lunch | 28.219298245614034 minutes


In [63]:
print(f"Far West | All time blocks | {average_time('Far West', 'all', PROMISE)} minutes") # Specified zones + all Time

Far West | All time blocks | 35.27167630057804 minutes


In [64]:
print(f"All zones | All time blocks | {average_time('all', 'all', PROMISE)} minutes") # all zones+ all Time

All zones | All time blocks | 31.193739635157545 minutes


# **Part I d**

I sorted all combinations by average delivery time.

In [65]:
average_sort = [] # give a list to save order delivery times
for z in ZONES:
  for tb in TIME_BLOCKS:
    avg = average_time(z,tb,PROMISE)
    average_sort.append((z,tb,avg))  # Calculate and save the average delivery time for the 12 combinations

def get_avg_time(item):
    return item[2]
average_sort.sort(key=get_avg_time,reverse=True) # get the avg time from list and list number from high to low

for item in average_sort:
    print(f"{item[0]} | {item[1]} | {item[2]} minutes") # show list

Far West | Fri/Sat eve | 42.9195238095238 minutes
North | Fri/Sat eve | 36.76533333333334 minutes
Far West | Weekday eve | 35.044117647058826 minutes
Central | Fri/Sat eve | 32.13263157894737 minutes
North | Weekday eve | 31.3756 minutes
Far West | Lunch | 30.58083333333333 minutes
Far West | Other | 30.04010416666667 minutes
Central | Weekday eve | 29.030909090909088 minutes
North | Lunch | 28.672631578947364 minutes
North | Other | 27.655624999999997 minutes
Central | Lunch | 26.798076923076923 minutes
Central | Other | 26.03684210526316 minutes


# **Part I e**

I think the late delivery rate is more useful for Rosa because it shows how often orders miss the promised time. Late orders cost her money through refunds and lost future orders, so this measure is more directly related to her decisions.
Far West / Fri/Sat eve has the highest late rate at 36.67% and the longest average delivery time at 42.92 minutes. The average is below 45 minutes, but more than one-third of orders are still late. If Rosa only looked at the average, she might underestimate the problem.
The two rankings do not always match. For example, Far West / Lunch has a slightly longer average delivery time than Far West / Other: 30.58 versus 30.04 minutes. However, its late rate is lower: 2.50% versus 4.69%. This shows that a longer average time does not always mean a higher late rate.
I would use the late rate to identify the zone and time-block combinations that need attention, and the average time to understand delivery speed. To choose the best promise, Rosa should also compare order volume and net profit.

# **Part II a**

Each late order costs 10 dollars in refunds and 16.20 dollars in lost future profit, for a total of 26.20 dollars.

In [66]:
def late_lost(costs):   # get to know how much lost when a late delivery happened
  return costs["refund"]+costs["churn_orders"]*costs["margin"] # Total cost = refund + lost future orders × profit per order

In [67]:
print(late_lost(COSTS)) # show how much per late

26.2


# **Part II b**



*   I used the average delivery time calculated earlier to determine the search range. Since the average represents only a general level—and actual delivery times can vary—I set aside half the space below the minimum average and half the space above the maximum average. In other words, I set the lower limit to 0.5 times the minimum average and the upper limit to 1.5 times the maximum average. This covers scenarios shorter and longer than the average time, broadening the search range, and then compares the net profit every minute.
*   The optimal time found this time did not fall on the boundary of the range, so I did not continue to expand the range, nor did I add code to automatically expand it. This result is the solution with the highest profit within the current search range.
*   I chose to test every minute rather than every 5 minutes because this allows for a more detailed comparison. For example, if I tested every 5 minutes, I might only compare the 40-minute and 45-minute marks, skipping the 41st, 42nd, 43rd, and 44th minutes in between. Using a 1-minute step size allows us to include these scenarios as well, reducing the likelihood of overlooking a more profitable option. Although this requires a few more calculations, the search range in this case is not very large.


In [68]:
def promise_time_range(average_sort):
  top_range =math.ceil(average_sort[0][2]/2 + average_sort[0][2]) # Use 1.5 times the highest average as the upper limit
  bottom_range =math.floor(average_sort[-1][2] - average_sort[-1][2]/2) # Use half the lowest average as the lower limit

  return range(bottom_range,top_range) # Try promises in 1 minute steps，the upper limit is excluded

In [70]:
print(promise_time_range(average_sort)) # test range

range(13, 65)


In [71]:
def best_promise(zones,time_blocks,promise_time_range,costs): # Start below any possible profit, including negative profits
  biggest_profit = float("-inf")
  best_time =promise_time_range[0]
  for p in promise_time_range: # Use a fixed seed to simulate delivery times
    all_times =delivery_times(zones,time_blocks,p,1)
    order_numbers =len(all_times)

    all_income = order_numbers * costs["margin"] # Calculate profit before deducting the cost of late order
    all_lost =np.sum(all_times > p) * late_lost(costs) # Total late cost = late orders × cost per late order
    net_profit =all_income - all_lost

    if net_profit > biggest_profit: # Save the promise with the highest net profit so far
      biggest_profit = net_profit
      best_time = p

  return best_time,float(biggest_profit)  # Return the best tested promise and its net profit

In [75]:
print(best_promise("Central","Lunch",promise_time_range(average_sort),COSTS)) # Test the result 1

(48, 2126.4)


In [76]:
print(best_promise("North","Weekday eve",promise_time_range(average_sort),COSTS)) # Test the result 2

(46, 1873.4)


In [78]:
promises = promise_time_range(average_sort) # Test if best time Within the select range

for z in ZONES:
    for tb in TIME_BLOCKS:
        best_time, best_profit = best_promise(z, tb, promises, COSTS)

        print(f"{z} | {tb} | {best_time} minutes | ${best_profit}")

        if best_time == promises[0] or best_time == promises[-1]: # if not show note
            print("Note: The best time is located at the search boundary.")

Central | Lunch | 48 minutes | $2126.4
Central | Weekday eve | 45 minutes | $2603.2
Central | Fri/Sat eve | 47 minutes | $1437.4
Central | Other | 43 minutes | $1606.8
North | Lunch | 47 minutes | $1542.2
North | Weekday eve | 46 minutes | $1873.4
North | Fri/Sat eve | 51 minutes | $1048.0
North | Other | 44 minutes | $1336.0
Far West | Lunch | 45 minutes | $1001.4
Far West | Weekday eve | 47 minutes | $1214.0
Far West | Fri/Sat eve | 52 minutes | $1252.4
Far West | Other | 48 minutes | $1498.0


In [ ]:
# the result shows it is all in the range that i selected

# **Part III: Streamlit App**
I used Claude and my skill to turn the notebook into a Streamlit app. Users can change the zone, time block, range, and costs. The Central / Lunch test matched my notebook: 48 minutes and $2,126.40.

- [GitHub repository](https://github.com/JingshuFu/722_Assignment1_Rosapizza)
- [Deployed Streamlit app](https://722-assignment1-rosapizza-jingshu.streamlit.app/)

# **Appendix: AI Use**
I used ChatGPT/Codex and Claude in VS Code to help complete this assignment.
ChatGPT/Codex helped me review, or revise the following parts:
- late_p: Told me to combining delivery arrays with np.concatenate.
- average_time: Helped adapt the late-percentage function to return average delivery time using .mean(). It also suggested a simpler version that selects zones and time blocks before looping
- late_lost: Reviewed my formula for the cost per late order.
- best_promise: Reviewed the net-profit calculation and suggested changing biggest_profit from 0 to float("-inf") so the function also works when all tested options have negative profits
It also helped fix my append error by changing append(z, tb, rate) to append((z, tb, rate)), corrected the syntax of my output strings, and explained the use of seed=1 for repeatable results.

ChatGPT/Codex helped draft the project-specific skill at .github/skills/notebook-to-streamlit/SKILL.md and the prompt asking Claude to use it.
Claude used the skill and my notebook to build the Streamlit app, creating app.py, logic.py, and requirements.txt.

ChatGPT/Codex compared my code and app against the assignment requirements. It also guided me through organizing the local project folder, creating a local Git repository, adding .gitignore file.

Examples of my prompts included:
“Does my code meet the requirements above?”
“Why is there an error here?”
“How do I set up the environment?”
“Is there a better solution?”

The main prompt sent to Claude was:
“Please use the notebook-to-streamlit skill at .github/skills/notebook-to-streamlit/SKILL.md to read the Jupyter Notebook in this project and convert its calculation logic into a Streamlit app.
Reuse my late_lost and best_promise functions. Keep seed=1 and the 1-minute search step. Import the provided starter package without recreating or modifying starter.py.

Follow the skill to create app.py, logic.py, and requirements.txt. Afterward, explain what each file does. Do not create a GitHub repository or deploy the app yet.”

I ran the notebook code and checked its outputs. I also tested the local app using Central / Lunch with candidate promises from 13 to 64 minutes and the original costs. It returned a recommended promise of 48 minutes and a net profit of $2,126.40, matching my notebook.

Part of the information and record for this appendix is also from and GPT